El primer paso es instalar en el entorno virtual ProyectoMasterIA creado en conda promp las librerias necesarias para la elaboración del ejercicio, que para este primer apartado son numpy para las operaciones matematicas, pandas para la limpieza de los datos, chardet para detectar la codificación de caracteres del archivo, csv para tener un control más manual del formato CSV y sqlite3  para trabajar con bases de datos SQLite en memoria 

In [1]:
import numpy as np
import pandas as pd
import chardet as ch
import csv
import sqlite3
print("ok")

ok


In [2]:
#Aqui se lee el archivo csv y se convierte en un dataframe de pandas
df=pd.read_csv('ventas_techventas.csv',
            sep=",", 
            encoding="ISO-8859-1", #con charset se detecto la codificació del archivo
            engine="python", #para indicar el uso del motor de python para leer el archivo
            quoting=csv.QUOTE_NONE, #se usa para ignorar las comillas dentro del dataset por comillas mal puestas en un producto
            parse_dates=["fecha"]) # Convierte automáticamente la columna fecha a formato de fecha.
df.head() #Muestra los primeros 5 registros

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,"""1003",2024-01-10,C003,DataSolutions,Centro,"Monitor 27""""",Electronica,5,350.0,0.05,"Ana Lï¿½ï¿"""
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz


In [3]:
#aqui se detecto la codificación del archivo
with open("ventas_techventas.csv", "rb") as f:
    resultado = ch.detect(f.read(100000))  

print(resultado)

{'encoding': 'ISO-8859-1', 'confidence': 0.73, 'language': ''}


Antes de realizar cualquier imputación o cambio en el archivo csv, se crea una copia independiente para poder modificar el archivo sin afectar el DataFrame original

In [4]:
venta= df.copy()
venta.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,"""1003",2024-01-10,C003,DataSolutions,Centro,"Monitor 27""""",Electronica,5,350.0,0.05,"Ana Lï¿½ï¿"""
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz


Aqui se realiza una limpieza de los datos en el DataFrame venta antes de analizarlos. Primero elimina todas las comillas dobles que puedan aparecer en los valores, lo cual suele ocurrir cuando los datos provienen de archivos CSV mal formateados o con texto encerrado en comillas y luego reemplaza todos los valores -1 por 0, ya que en muchos conjuntos de datos el valor -1 se utiliza como indicador de error o dato faltante, y Ana Lopez de imputa manualmente debido a que su nombre esta mal codificado en el archivo con carácteres dañinos

In [5]:
venta= venta.replace('"', '', regex=True)
venta= venta.replace(-1, 0, regex=True)
venta= venta.replace("Ana Lï¿½ï¿", "Ana Lopez", regex=True)
venta.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lopez
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,1003,2024-01-10,C003,DataSolutions,Centro,Monitor 27,Electronica,5,350.0,0.05,Ana Lopez
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz


Se hace una exploración rápida del DataFrame  para entender cómo están estructurados los datos antes de analizarlos.

In [6]:
#Para ver los tipos de datos
venta.dtypes

order_id                      str
fecha              datetime64[us]
cliente_id                    str
cliente_nombre                str
region                        str
producto                      str
categoria                     str
cantidad                    int64
precio_unitario           float64
descuento                 float64
vendedor                      str
dtype: object

# Interpretación de venta.info()
El DataFrame contiene 60 registros y 11 columnas, lo que indica que se trata de un conjunto de datos pequeño donde la mayoría de las columnas son de tipo texto (str), como el identificador de la orden, cliente, región, producto, categoría y vendedor, lo que sugiere que el dataset está orientado a describir información comercial y contextual de cada venta.En términos de calidad de datos, casi todas las columnas tienen 60 valores no nulos, excepto producto, que tiene 59, lo que indica la presencia de un valor faltante que podría requerir tratamiento, con respecto al uso de memoria es bajo (5.3 KB), lo que confirma que es un dataset liviano y fácil de procesar.

In [7]:
venta.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         60 non-null     str           
 1   fecha            60 non-null     datetime64[us]
 2   cliente_id       60 non-null     str           
 3   cliente_nombre   60 non-null     str           
 4   region           60 non-null     str           
 5   producto         59 non-null     str           
 6   categoria        60 non-null     str           
 7   cantidad         60 non-null     int64         
 8   precio_unitario  60 non-null     float64       
 9   descuento        60 non-null     float64       
 10  vendedor         60 non-null     str           
dtypes: datetime64[us](1), float64(2), int64(1), str(7)
memory usage: 5.3 KB


In [8]:
#para ver estadisticas descriptivas
venta.describe()

,fecha,cantidad,precio_unitario,descuento
count,60,60.000000,60.000000,60.000000
mean,2024-03-27 00:24:00,9.783333,378.250000,0.055000
min,2024-01-05 00:00:00,0.000000,25.000000,0.000000
25%,2024-02-14 06:00:00,3.000000,92.500000,0.000000
50%,2024-03-26 12:00:00,5.500000,120.000000,0.050000
75%,2024-05-07 18:00:00,15.750000,350.000000,0.100000
max,2024-06-18 00:00:00,35.000000,1200.000000,0.200000
std,NaN,8.837692,448.815602,0.058004


In [9]:
# saber cuantos nulos hay en la base de datos por columna

print("cantidad de nulos")
print(f" {venta.isna().sum()} ")

cantidad de nulos
 order_id           0
fecha              0
cliente_id         0
cliente_nombre     0
region             0
producto           1
categoria          0
cantidad           0
precio_unitario    0
descuento          0
vendedor           0
dtype: int64 


In [10]:
#identifica la fila que contiene el nulo
venta[venta['producto'].isna()]

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
42,1043,2024-05-01,C010,DeltaOps,Norte,NaN,Electronica,2,350.0,0.05,Maria Torres


In [11]:
#Se trata el nulo imputandolo con la palabra desconocido ya que indica explícitamente que la información del producto no está disponible o no fue registrada  
venta["producto"] = venta["producto"].fillna("Desconocido")
print(f" {venta.isna().sum()} ")

 order_id           0
fecha              0
cliente_id         0
cliente_nombre     0
region             0
producto           0
categoria          0
cantidad           0
precio_unitario    0
descuento          0
vendedor           0
dtype: int64 


In [12]:
#Se verifica que la base de datos no tenga valores duplicados
venta.duplicated().sum()

np.int64(0)

In [13]:
#Se  extrae información de la columna de fecha y crea nuevas columnas para analizar mejor el tiempo en las ventas

venta['anio']       = venta['fecha'].dt.year         # Año numérico, se crea la columna
venta['mes']        = venta['fecha'].dt.month        # Mes numérico (1–12)
venta['dia']        = venta['fecha'].dt.day          # Día del mes
venta['nombre_mes'] = venta['fecha'].dt.month_name() # Nombre del mes
venta['dia_semana'] = venta['fecha'].dt.day_name()  

venta[['fecha','anio','mes','nombre_mes','dia','dia_semana']].head(5)

,fecha,anio,mes,nombre_mes,dia,dia_semana
0,2024-01-05,2024,1,January,5,Friday
1,2024-01-07,2024,1,January,7,Sunday
2,2024-01-10,2024,1,January,10,Wednesday
3,2024-01-12,2024,1,January,12,Friday
4,2024-01-15,2024,1,January,15,Monday


# Ejercicio 2

Se crea una base de datos en memoria y se guarda el DataFrame venta como una tabla dentro de esa base de datos.

In [14]:
conn = sqlite3.connect(':memory:')

# Exportar el DataFrame a SQLite
venta.to_sql("ventas", conn, if_exists="replace", index=False)

60

In [15]:
#Se consulta la base de datos SQLite que se creo en memoria y se trae todos los registros de la tabla

bd=conn.cursor() #este metodo permite hacer la conexion directa a la consulta
bd.execute("""
SELECT * FROM ventas
            """)

resultado=bd.fetchone()
print(resultado)


('1001', '2024-01-05 00:00:00', 'C001', 'TechCorp SA', 'Norte', 'Laptop Pro 15', 'Electronica', 2, 1200.0, 0.1, 'Ana Lopez', 2024, 1, 5, 'January', 'Friday')


In [16]:
print("Lista todos los productos únicos disponibles usando un alias de columna descriptivo")


pd.read_sql("""
SELECT DISTINCT producto AS producto_disponible
FROM ventas
ORDER BY producto ASC

            """,conn)


Lista todos los productos únicos disponibles usando un alias de columna descriptivo


,producto_disponible
0,Desconocido
1,Laptop Pro 15
2,Monitor 27
3,Mouse Inalambrico
4,Router WiFi 6
5,SSD 1TB
6,Teclado Mecanico


In [17]:
print("Pedidos del primer trimestre mayor a 5 unidades")

pd.read_sql("""
SELECT 
order_id,DATE(fecha) AS fecha,producto,cantidad,precio_unitario
FROM ventas
WHERE fecha BETWEEN '2024-01-01' AND '2024-03-31'
  AND cantidad > 5;
""", conn)



Pedidos del primer trimestre mayor a 5 unidades


,order_id,fecha,producto,cantidad,precio_unitario
0,1002,2024-01-07,Mouse Inalambrico,15,25.0
1,1004,2024-01-12,Teclado Mecanico,10,85.0
2,1006,2024-01-18,SSD 1TB,20,95.0
3,1008,2024-01-22,Router WiFi 6,8,120.0
4,1009,2024-01-25,SSD 1TB,12,95.0
5,1012,2024-02-05,Mouse Inalambrico,30,25.0
6,1015,2024-02-12,SSD 1TB,25,95.0
7,1017,2024-02-18,Monitor 27,6,350.0
8,1018,2024-02-20,SSD 1TB,18,95.0
9,1019,2024-02-22,Mouse Inalambrico,12,25.0


In [18]:
print("Vendedores cuyo ingreso bruto total (cantidad × precio) supera $10,000 USD.")

pd.read_sql("""
SELECT 
    vendedor,
    SUM(cantidad * precio_unitario) AS ingreso_bruto_total
FROM ventas
GROUP BY vendedor
HAVING SUM(cantidad * precio_unitario) > 10000
ORDER BY ingreso_bruto_total DESC;""",conn)

Vendedores cuyo ingreso bruto total (cantidad × precio) supera $10,000 USD.


,vendedor,ingreso_bruto_total
0,Ana Lopez,29710.0
1,Luis Mora,21930.0
2,Carlos Ruiz,21355.0
3,Maria Torres,16410.0


In [19]:
print("Por categoría: total de pedidos, suma de unidades vendidas y precio unitario promedio.")

pd.read_sql("""
SELECT 
    categoria,
    COUNT(order_id) AS total_pedidos,
    SUM(cantidad) AS unidades_vendidas,
    AVG(precio_unitario) AS precio_unitario_promedio
FROM ventas
GROUP BY categoria
ORDER BY unidades_vendidas DESC;
""", conn)

Por categoría: total de pedidos, suma de unidades vendidas y precio unitario promedio.


,categoria,total_pedidos,unidades_vendidas,precio_unitario_promedio
0,Almacenamiento,12,260,95.0
1,Perifericos,15,215,53.0
2,Electronica,25,73,792.0
3,Redes,8,39,120.0


In [20]:
# print("Crea la tabla vendedores (abajo) y calcula el % de cumplimiento de meta de cada uno.")
#Se borra la tabla vendedores si existe con drop table para que no de error al volver a ejecutar y se crea nuevamente
conn.executescript ("""
DROP TABLE IF EXISTS vendedores;
                    
--Tabla:vendedores

CREATE TABLE vendedores(
    vendedor TEXT,
    zona TEXT,
    meta REAL 
            );
                    
INSERT INTO vendedores VALUES
("Ana Lopez","Norte",8000),
("Carlos Ruiz","Sur",7500),
("Luis Mora","Norte",8000),
("Maria Torres","Centro",7000);      
""");
print("Tabla vendedores creada con éxito")





Tabla vendedores creada con éxito


In [21]:
# Una vez creada la tabla clientes, se calcula cuánto dinero ha generado cada vendedor, se compara con su meta de ventas y se obtiene el porcentaje de cumplimiento de esa meta.

conn.executescript ("""
SELECT 
    v.vendedor,
    v.meta,
    SUM(ventas.cantidad * ventas.precio_unitario) AS ingreso_real,
    ROUND((SUM(ventas.cantidad * ventas.precio_unitario) / v.meta) * 100, 2) AS porcentaje_cumplimiento
FROM vendedores v -- a vendedores se les pone el alias v
JOIN ventas --se usa Join porque ambas tablas tienen la columna en común vendedor y permite relacionar las tablas
ON v.vendedor = ventas.vendedor
GROUP BY v.vendedor, v.meta;""")

pd.read_sql("""
SELECT 
    v.vendedor,
    v.meta,
    SUM(ventas.cantidad * ventas.precio_unitario) AS ingreso_real,
    ROUND((SUM(ventas.cantidad * ventas.precio_unitario) / v.meta) * 100, 2) AS porcentaje_cumplimiento
FROM vendedores v
JOIN ventas
ON v.vendedor = ventas.vendedor
GROUP BY v.vendedor, v.meta;
""", conn)


,vendedor,meta,ingreso_real,porcentaje_cumplimiento
0,Ana Lopez,8000.0,29710.0,371.38
1,Carlos Ruiz,7500.0,21355.0,284.73
2,Luis Mora,8000.0,21930.0,274.13
3,Maria Torres,7000.0,16410.0,234.43


In [22]:
print("Cliente con mayor ingreso total en el primer semestre")

pd.read_sql("""
SELECT 
    cliente_nombre,
    SUM(cantidad * precio_unitario) AS ingreso_total
FROM ventas
WHERE fecha BETWEEN '2024-01-01' AND '2024-06-30'
GROUP BY cliente_nombre
ORDER BY ingreso_total DESC
LIMIT 1
""", conn)


Cliente con mayor ingreso total en el primer semestre


,cliente_nombre,ingreso_total
0,TechCorp SA,20425.0


# Ejercicio 3

In [23]:
#Crear columnas ingreso_bruto, ingreso_neto y mes. Agrupa por mes con groupby + agg.
venta["fecha"] = pd.to_datetime(df["fecha"]) #convierte la fecha en formato fecha

#Se crea la columna ingreso bruto multiplicando la columna cantidad * precio unitario
venta["ingreso_bruto"] = venta["cantidad"] * venta["precio_unitario"]

#Se crea la columna ingreso neto que es multiplicar el ingreso bruto por el descuento, descuento que sale de restar 1(que representa el 100%) menos el porcentaje de descuento

venta["ingreso_neto"] = venta["ingreso_bruto"] * (1 - venta["descuento"])


In [24]:

#Aqui se agrupa todas las ventas de un mismo mes y calcula estadísticas para ese grupo resumiendo la información y guardandola en un nuevo dataframe de resumen mensual
resumen_mensual = (
    venta.groupby("mes")
      .agg( #agg es para agregar las columnas a mostrar
          total_ventas=("cantidad", "sum"),
          ingreso_bruto=("ingreso_bruto", "sum"),
          ingreso_neto=("ingreso_neto", "sum"),
          precio_promedio=("precio_unitario", "mean"),
        
      )
      .reset_index()
)

print(resumen_mensual)

   mes  total_ventas  ingreso_bruto  ingreso_neto  precio_promedio
0    1            78        14875.0      13773.50       472.000000
1    2           107        14220.0      13050.00       354.500000
2    3           104        15605.0      14454.00       338.636364
3    4           113        18360.0      16689.00       333.181818
4    5           103        13965.0      12587.25       431.363636
5    6            82        12380.0      11209.25       327.857143


In [25]:
#Se crea una nueva tabla metas que contiene los meses y la meta por mes

metas = pd.DataFrame({
    "mes": [1,2,3,4,5,6],
    "meta_ventas": [200, 100, 120, 120,100,200]
})

In [26]:
#Se une la tabla metas con el dataframe resumen mensual hecho anteriormente
resumen_final = resumen_mensual.merge(
    metas,
    on="mes",
    how="left"
)

In [27]:
#A esa nueva tabla de resumen final, se calcula el % de cumplimiento diviendo la cantidad de ventas hechas sobre la cantidad de ventas esperadas como meta
resumen_final["%_cumplimiento"] = (
    resumen_final["total_ventas"] / resumen_final["meta_ventas"]
) * 100

In [28]:
resumen_final

,mes,total_ventas,ingreso_bruto,ingreso_neto,precio_promedio,meta_ventas,%_cumplimiento
0,1,78,14875.0,13773.50,472.000000,200,39.000000
1,2,107,14220.0,13050.00,354.500000,100,107.000000
2,3,104,15605.0,14454.00,338.636364,120,86.666667
3,4,113,18360.0,16689.00,333.181818,120,94.166667
4,5,103,13965.0,12587.25,431.363636,100,103.000000
5,6,82,12380.0,11209.25,327.857143,200,41.000000


In [29]:
#se exporta con to_sql y se verifica leyendo de vuelta la tabla resumen final

resumen_final.to_sql('ventas_finales',    conn, if_exists='replace', index=False)



#Se verifican todas las tablas que se tienen en la base de datos al momento

tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn) 
print("Tablas disponibles en la base de datos SQLite:")
print(tablas.to_string(index=False))

Tablas disponibles en la base de datos SQLite:
          name
        ventas
    vendedores
ventas_finales


In [32]:
#Se realiza una consulta SQL para contar cuántos registros  tiene la tabla ventas_finales y muestra el resultado.

conteo = pd.read_sql("SELECT COUNT(*) AS total_registros FROM ventas_finales", conn)
print(conteo)

   total_registros
0                6
